<table class="table table-bordered">
    <tr>
        <th style="text-align:center; width:25%"><img src='https://www.np.edu.sg/PublishingImages/Pages/default/odp/ICT.jpg' style="width: 250px; height: 125px; "></th>
        <th style="text-align:center;"><h1>Deep Learning</h1><h2>Assignment 2 - Character Generator Model (Problem 2)</h2><h3>AY2020/21 Semester</h3></th>
    </tr>
</table>

In [2]:
# Import the Required Packages
# Enter your code here:
from tensorflow import keras
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras import regularizers
from tensorflow.keras.layers import GRU
from keras.regularizers import l2
from keras.regularizers import l1
import os
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout
import numpy as np
import random

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

Using TensorFlow backend.


## Step 1 – Data Loading and Processing

### 1.1 Data Loading

In [3]:
# read in the text file, transforming everything to lower case
text = open('holmes.txt').read().lower()
print('The original text has ' + str(len(text)) + ' characters.\n')

The original text has 562439 characters.



### 1.2 Data Processing


In [4]:
### print out the first 1000 characters of the raw text to get a sense of what characters to remove
text[:2000]

"ï»¿the adventures of sherlock holmes by sir arthur conan doyle\n\n   i. a scandal in bohemia\n  ii. the red-headed league\n iii. a case of identity\n  iv. the boscombe valley mystery\n   v. the five orange pips\n  vi. the man with the twisted lip\n vii. the adventure of the blue carbuncle\nviii. the adventure of the speckled band\n  ix. the adventure of the engineer's thumb\n   x. the adventure of the noble bachelor\n  xi. the adventure of the beryl coronet\n xii. the adventure of the copper beeches\n\n\nadventure i. a scandal in bohemia\n\ni.\n\nto sherlock holmes she is always the woman. i have seldom heard\nhim mention her under any other name. in his eyes she eclipses\nand predominates the whole of her sex. it was not that he felt\nany emotion akin to love for irene adler. all emotions, and that\none particularly, were abhorrent to his cold, precise but\nadmirably balanced mind. he was, i take it, the most perfect\nreasoning and observing machine that the world has seen, but as a\

In [5]:
# remove all '\n' and '\r' from text
text = text.replace('\n',' ') 
text = text.replace('\r',' ')

In [6]:
# create a function 'clean_text' to clean text so that only the following letters and punctation remain
def clean_text(text):
    punctuation = ['!', ',', '.', ':', ';', '?', '-', "'",' ']
    letters='abcdefghijklmnopqrstuvwxyz'
    letterList=list(letters)
    
    # Enter your code here:
    for character in text:
        if (character in punctuation) or (character in letterList):
            continue
        else:
            text = text.replace(character, "")
    
    text = ' '.join(text.split())
    
    return text

In [7]:
# clean data using clean_text function
text = clean_text(text)
text[:2000]

In [11]:
# count the number of unique characters in the text
chars = sorted(list(set(text)))
# print some of the text, as well as statistics
print ("This document has " +  str(len(text)) + " total number of characters.")
print ("This document has " +  str(len(chars)) + " unique characters.")

<class 'spacy.tokens.doc.Doc'>
This document has 122813 total number of characters.
This document has 122813 unique characters.


In [ ]:
# create a function 'generate_text_io' to generate text inputs based on window_size and the corresponding labels
def generate_text_io(text, window_size):
    inputs = [] # store inputs
    labels = [] # stores label
    
    # Enter your code here:
    maxlen = window_size
    step = 1
    
    for i in range(0, len(text) - maxlen, step):
        inputs.append(text[i : i+ maxlen])
        labels.append(text[i + maxlen])
        
    return (inputs, labels)

In [ ]:
# this dictionary is a function mapping each unique character to a unique integer
chars_to_indices = dict((c, i) for i, c in enumerate(chars))  # map each unique character to unique integer

# this dictionary is a function mapping each unique integer back to a unique character
indices_to_chars = dict((i, c) for i, c in enumerate(chars))  # map each unique integer back to unique character

In [ ]:
# create a function 'encode_io_pairs' to perform one-hot encoding of inputs and labels
def encode_io_pairs(text,window_size): # window_size determines # of characters in each input
    
    # Enter your code here:
    inputs, labels = generate_text_io(text, window_size)
    
    x = np.zeros((len(inputs), window_size, len(chars)), dtype=np.bool)
    y = np.zeros((len(inputs), len(chars)), dtype=np.bool)
    for i, sentence in enumerate(inputs):
        for t, char in enumerate(sentence):
            x[i, t, chars_to_indices[char]] = 1
        y[i, chars_to_indices[labels[i]]] = 1    
    
    return x, y

In [ ]:
# perform one-hot encoding of inputs and labels
window_size = 100
X, y = encode_io_pairs(text, window_size)

### 1.3 Splitting Dataset into Inputs (X) and Labels (y)

In [ ]:
# Note: You may choose to perform this step before encoding the data (step 1.2).
# Enter your code here:
print(X.shape)
print(y.shape)

## Step 2 – Develop a Character Generator Model

### Model #1 (Replicate as necessary for other models)

In [ ]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [ ]:
# Build the Model
# Enter your code here:
model = models.Sequential()
model.add(layers.LSTM(256, return_sequences=True, input_shape=(window_size, len(chars))))
model.add(Dropout(0.2))
model.add(layers.LSTM(256))
model.add(Dropout(0.2))
model.add(layers.Dense(len(chars), activation='softmax'))
model.summary()

In [ ]:
# Train the Model
# Enter your code here:
optimizer = optimizers.RMSprop(lr=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

for epoch in range(1, 20):
    print('epoch', epoch)
    # Fit the model for 1 epoch on the available training data
    model.fit(X, y,
              batch_size=128,
              epochs=1)

    # Select a text seed at random
    start_index = random.randint(0, len(text) - window_size - 1)
    generated_text = text[start_index: start_index + window_size]
#     generated_text = input("Enter text to start predicting from: ")
    print('--- Generating with seed: "' + generated_text + '"')

    for temperature in [0.2, 0.5, 1.0, 1.2]:
        print('------ temperature:', temperature)
        sys.stdout.write(generated_text)

        # We generate 400 characters
        for i in range(400):
            sampled = np.zeros((1, window_size, len(chars)))
            for t, char in enumerate(generated_text):
                sampled[0, t, chars_to_indices[char]] = 1.

            preds = model.predict(sampled, verbose=0)[0]
            next_index = sample(preds, temperature)
            next_char = chars[next_index]

            generated_text += next_char
            generated_text = generated_text[1:]

            sys.stdout.write(next_char)
            sys.stdout.flush()
        print()

In [ ]:
# Save the Model
model.save('chgen_model_1.h5')

### Model #2 (Replicate as necessary for other models)

In [ ]:
# Build the Model
# Enter your code here:
from transformers import GPT2Tokenizer, TFGPT2Model
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = TFGPT2Model.from_pretrained('gpt2')

In [ ]:
# Train the Model
# Enter your code here:


In [ ]:
# Save the Model
model.save('chgen_model_2.h5')

## Step 3 – Use the Best Model to make prediction

In [ ]:
model.load_weights('chgen_model_best.h5')

In [ ]:
# takes the user input
text_input = np.array([input()])

In [ ]:
# one-hot encode the user input
# Enter your code here:


In [ ]:
# show the model output using predict function
# Enter your code here:
